In [ ]:
#  Copyright (c) Microsoft Corporation.
#  版权所有 (c) 微软公司。
#  Licensed under the MIT License.
#  根据 MIT 许可证授权。

# Introduction
# 介绍

Though users can automatically run the whole Quant research worklfow based on configurations with Qlib.
虽然用户可以基于配置利用 Qlib 自动运行整个量化研究工作流。

Some advanced users usally would like to carefully customize each component to explore more in Quant.
一些高级用户通常希望仔细定制每个组件，以在量化领域进行更多探索。

If you just want a simple example of Qlib. [Quick start](https://github.com/microsoft/qlib#quick-start) and [workflow_by_code](https://github.com/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb) may be a better choice for you.
如果您只想要一个简单的 Qlib 示例，[快速入门](https://github.com/microsoft/qlib#quick-start) 和 [代码工作流](https://github.com/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb) 对您来说可能是更好的选择。

If you want to know more details about Quant research, this notebook may be a better place for you to start.
如果您想了解有关量化研究的更多细节，本笔记本（notebook）可能是您更好的起点。

We hope this script could be a tutorial for users who are interested in the details of Quant.
我们希望这个脚本可以作为对量化细节感兴趣的用户的教程。

This notebook tries to demonstrate how can we use Qlib to build components step by step.
本笔记本试图演示如何使用 Qlib 逐步构建各个组件。

In [ ]:
from pprint import pprint
from pathlib import Path
import pandas as pd

In [ ]:
MARKET = "csi300"
BENCHMARK = "SH000300"
EXP_NAME = "tutorial_exp"

# Data
# 数据

## Get data
## 获取数据

Users can follow [the steps](https://github.com/microsoft/qlib/tree/main/scripts#download-qlib-data) to download data with CLI.
用户可以按照[这些步骤](https://github.com/microsoft/qlib/tree/main/scripts#download-qlib-data)使用命令行界面（CLI）下载数据。

In this example we use the underlying API to automatically download data
在本示例中，我们使用底层 API 来自动下载数据。

In [ ]:
from qlib.tests.data import GetData

GetData().qlib_data(exists_skip=True)

In [ ]:
import qlib

qlib.init()

## Inspect raw data
## 检查原始数据

Currently, Qlib support several kinds of data source.
目前，Qlib 支持几种类型的数据源。

### Calendar
### 交易日历

In [ ]:
from qlib.data import D

print(D.calendar(start_time="2010-01-01", end_time="2017-12-31", freq="day")[:2])  # calendar data / 日历数据

### Basic data
### 基础数据

In [ ]:
df = D.features(
    ["SH601216"],
    ["$open", "$high", "$low", "$close", "$factor"],
    start_time="2020-05-01",
    end_time="2020-05-31",
)

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"
fig = go.Figure(
    data=[
        go.Candlestick(
            x=df.index.get_level_values("datetime"),
            open=df["$open"],
            high=df["$high"],
            low=df["$low"],
            close=df["$close"],
        )
    ]
)
fig.show()

### price adjustment
### 价格复权

Maybe you think the price is not what it looks like in real world.
也许你认为价格与现实世界中的价格有所不同。

Due to the price adjustment, the price will be different from the real trading data .
由于价格复权，该价格会与实际的交易数据不同。

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Candlestick(
            x=df.index.get_level_values("datetime"),
            open=df["$open"] / df["$factor"],
            high=df["$high"] / df["$factor"],
            low=df["$low"] / df["$factor"],
            close=df["$close"] / df["$factor"],
        )
    ]
)
fig.show()

Please notice the price gap on [2020-05-26](http://vip.stock.finance.sina.com.cn/corp/view/vISSUE_ShareBonusDetail.php?stockid=601216&type=1&end_date=2020-05-20)
请注意 [2020-05-26](http://vip.stock.finance.sina.com.cn/corp/view/vISSUE_ShareBonusDetail.php?stockid=601216&type=1&end_date=2020-05-20) 的价格缺口。

If we want to represent the change of assets value by price, adjust prices are necesary.
如果想用价格来表示资产价值的变化，复权价是必要的。

By default, Qlib stores the adjusted prices.
默认情况下，Qlib 存储的是复权价格。

### Static universe V.S. dynamic universe
### 静态股票池与动态股票池

Dynamic universe
动态股票池

In [ ]:
# dynamic universe
# 动态股票池
universe = D.list_instruments(D.instruments("csi100"), start_time="2010-01-01", end_time="2020-12-31")
pprint(universe)

In [ ]:
print(len(universe))

Qlib use dynamic universe by default.
Qlib 默认使用动态股票池。

csi100 has around 100 stocks each day(it is not that accurate due to the low precision of data).
沪深100指数每天包含大约100只股票（由于数据精度较低，该结果并不是非常精确）。

In [ ]:
df = D.features(D.instruments("csi100"), ["$close"], start_time="2010-01-01", end_time="2020-12-31")
df.groupby("datetime").size().plot()

### Point-In-Time data
### 点点对应数据（PIT数据）

#### download data
#### 下载数据

NOTE: To run the test faster, we only download the data of two stocks
注意：为了加快测试运行速度，我们仅下载两只股票的数据。

In [ ]:
p = Path("~/.qlib/qlib_data/cn_data/financial").expanduser()

In [ ]:
if not p.exists():
    !cd ../../scripts/data_collector/pit/ && pip install -r requirements.txt
    !cd ../../scripts/data_collector/pit/ && python collector.py download_data --source_dir ~/.qlib/stock_data/source/pit --start 2000-01-01 --end 2020-01-01 --interval quarterly --symbol_regex "^(600519|000725).*"
    !cd ../../scripts/data_collector/pit/ && python collector.py normalize_data --interval quarterly --source_dir ~/.qlib/stock_data/source/pit --normalize_dir ~/.qlib/stock_data/source/pit_normalized
    !cd ../../scripts/ && python dump_pit.py dump --csv_path ~/.qlib/stock_data/source/pit_normalized --qlib_dir ~/.qlib/qlib_data/cn_data --interval quarterly

#### querying data
#### 查询数据

using `roewa(performanceExpressROEWa,业绩快报净资产收益率ROE-加权)` as an example
以 `roewa`（加权业绩快报净资产收益率）为例。

If we want to get fundamental data `in the most recent quarter` daily, we can use following example.
如果我们想每天获取“最近一个季度”的基本面数据，可以使用以下示例。

Maitai release part of its fundamental data on [2019-07-13](http://www.cninfo.com.cn/new/disclosure/detail?stockCode=600519&announcementId=1206443183&orgId=gssh0600519&announcementTime=2019-07-13) and  release others on [2019-07-18](http://www.cninfo.com.cn/new/disclosure/detail?stockCode=600519&announcementId=1206456129&orgId=gssh0600519&announcementTime=2019-07-18)
茅台在 [2019-07-13](http://www.cninfo.com.cn/new/disclosure/detail?stockCode=600519&announcementId=1206443183&orgId=gssh0600519&announcementTime=2019-07-13) 披露了部分基本面数据，并在 [2019-07-18](http://www.cninfo.com.cn/new/disclosure/detail?stockCode=600519&announcementId=1206456129&orgId=gssh0600519&announcementTime=2019-07-18) 披露了其余数据。

In [ ]:
instruments = ["sh600519"]
data = D.features(
    instruments,
    ["P($$roewa_q)"],
    start_time="2019-01-01",
    end_time="2019-07-19",
    freq="day",
)

In [ ]:
data.tail(15)

### experss engine
### 表达式引擎

In [ ]:
D.features(
    ["sh600519"],
    ["(EMA($close, 12) - EMA($close, 26))/$close - EMA((EMA($close, 12) - EMA($close, 26))/$close, 9)/$close"],
)

## Dataset loading and preprocessing
## 数据集加载与预处理

Some heuristic principles of create features
创建特征的一些启发式原则：
- make the features comparable between instrumets: remove unit from the features.
- 使不同股票之间的特征具有可比性：去除特征的单位。
- try to keep the distribution invariant
- 尽量保持分布不变。
- keep the scale of features similar
- 保持特征的尺度（scale）相似。

### data loader
### 数据加载器

It's interface can be found [here](https://github.com/microsoft/qlib/blob/main/qlib/data/dataset/loader.py#L24) 
其接口可以在[此处](https://github.com/microsoft/qlib/blob/main/qlib/data/dataset/loader.py#L24)找到。

QlibDataLoader is an implementation which load data from Qlib's data source
QlibDataLoader 是从 Qlib 数据源加载数据的具体实现。

In [ ]:
from qlib.data.dataset.loader import QlibDataLoader

In [ ]:
qdl = QlibDataLoader(config=(["$close / Ref($close, 10)"], ["RET10"]))

In [ ]:
qdl.load(instruments=["sh600519"], start_time="20190101", end_time="20191231")

### data handler
### 数据处理器

finance data can't be perfect.
金融数据不可能完美。

We have to process them before feeding them into Models
在将它们喂给模型之前，我们必须对它们进行处理。

In [ ]:
df = qdl.load(instruments=["sh600519"], start_time="20190101", end_time="20191231")

In [ ]:
df.isna().sum()

In [ ]:
df.plot(kind="hist")

Datahander is responsible for data preprocessing and provides data fetching interface
DataHandler 负责数据预处理，并提供数据获取接口。

In [ ]:
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.dataset.processor import ZScoreNorm, Fillna

In [ ]:
# NOTE: normally, the training & validation time range will be  `fit_start_time` ， `fit_end_time`
# 注意：通常情况下，训练与验证的时间范围为 `fit_start_time` 到 `fit_end_time`
# however，all the components are decomposed, so the training & validation time range is unknown when preprocessing.
# 然而，所有的组件都是解耦的，因此预处理时训练与验证的时间范围是未知的。
dh = DataHandlerLP(
    instruments=["sh600519"],
    start_time="20170101",
    end_time="20191231",
    infer_processors=[
        ZScoreNorm(fit_start_time="20170101", fit_end_time="20181231"),
        Fillna(),
    ],
    data_loader=qdl,
)

In [ ]:
df = dh.fetch()

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
df.plot(kind="hist")

### dataset
### 数据集

#### basic dataset
#### 基础数据集

In [ ]:
from qlib.data.dataset import DatasetH, TSDatasetH

In [ ]:
ds = DatasetH(dh, segments={"train": ("20180101", "20181231"), "valid": ("20190101", "20191231")})

In [ ]:
ds.prepare("train")

In [ ]:
ds.prepare("valid")

#### Time Series Dataset
#### 时间序列数据集

For different model, the required dataset format will be different.
对于不同的模型，所需的数据集格式也会不同。

For example, Qlib provides a Time Series Dataset(TSDatasetH) to help users to create time-series dataset.
例如，Qlib 提供了一个时间序列数据集（TSDatasetH）来帮助用户创建时间序列数据集。

In [ ]:
ds = TSDatasetH(
    step_len=10,
    handler=dh,
    segments={"train": ("20180101", "20181231"), "valid": ("20190101", "20191231")},
)
train_sampler = ds.prepare("train")

In [ ]:
train_sampler

In [ ]:
train_sampler[0]  # Retrieving the first example / 检索第一个样本

In [ ]:
train_sampler["2018-01-08", "sh600519"]  # get the time series by <'timestamp', 'instrument_id'> index / 通过 <'timestamp', 'instrument_id'> 索引获取时间序列

### Off-the-shelf dataset
### 开箱即用的数据集

Qlib integrated some dataset alreadly
Qlib 已经集成了一些数据集。

In [ ]:
handler_kwargs = {
    "start_time": "2008-01-01",
    "end_time": "2020-08-01",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": MARKET,
}
handler_conf = {
    "class": "Alpha158",
    "module_path": "qlib.contrib.data.handler",
    "kwargs": handler_kwargs,
}
pprint(handler_conf)

In [ ]:
from qlib.utils import init_instance_by_config

In [ ]:
hd = init_instance_by_config(handler_conf)

Using config to create instance is a highly frequently used practice in Qlib (e.g. the [workflows configurations](https://github.com/microsoft/qlib/blob/main/examples/benchmarks/LightGBM/workflow_config_lightgbm_Alpha158.yaml) are based on it).
使用配置创建实例在 Qlib 中是一种非常高频的做法（例如，[工作流配置](https://github.com/microsoft/qlib/blob/main/examples/benchmarks/LightGBM/workflow_config_lightgbm_Alpha158.yaml)就是基于此实现的）。


The above configuration is the same as the code below
上述配置与下面的代码等价：

In [ ]:
from qlib.contrib.data.handler import Alpha158

hd = Alpha158(**handler_kwargs)

This dataset has the same structure as the simple one with 1 column  we created just now.
该数据集的结构与我们刚才创建的只有一列的简单数据集相同。

In [ ]:
df = hd.fetch()

In [ ]:
df

In [ ]:
hd.data_loader

In [ ]:
hd.data_loader.fields

#### some details
#### 一些细节

The training data may not be the same as the test data.
训练数据与测试数据可能不同。

e.g.
例如：
- the training dataset and test dataset use a different fitlering rules,  data processing
- 训练数据集和测试数据集使用不同的过滤规则、数据处理方式

In [ ]:
hd.learn_processors

In [ ]:
hd.infer_processors

In [ ]:
hd.process_type  # appending type / 拼接类型

In [ ]:
hd.fetch(col_set="label", data_key=hd.DK_L)

In [ ]:
hd.fetch(col_set="label", data_key=hd.DK_I)

In [ ]:
dataset_conf = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": hd,
        "segments": {
            "train": ("2008-01-01", "2014-12-31"),
            "valid": ("2015-01-01", "2016-12-31"),
            "test": ("2017-01-01", "2020-08-01"),
        },
    },
}

In [ ]:
dataset = init_instance_by_config(dataset_conf)

# Model Training & Inference
# 模型训练与推理

[Model interface](https://github.com/microsoft/qlib/blob/main/qlib/model/base.py)
[模型接口](https://github.com/microsoft/qlib/blob/main/qlib/model/base.py)

In [ ]:
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord, SigAnaRecord

In [ ]:
model = init_instance_by_config(
    {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    }
)

In [ ]:
# start exp to train model
# 启动实验以训练模型
with R.start(experiment_name=EXP_NAME):
    model.fit(dataset)
    R.save_objects(trained_model=model)

    rec = R.get_recorder()
    rid = rec.id  # save the record id / 保存记录 ID

    # Inference and saving signal
    # 推理并保存信号
    sr = SignalRecord(model, dataset, rec)
    sr.generate()

# Evaluation:
# 评估：
- Signal-based
- 基于信号的评估
- Portfolio-based: backtest
- 基于投资组合的评估：回测

In [ ]:
###################################
# prediction, backtest & analysis
# 预测、回测与分析
###################################
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "signal": "<PRED>",
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": "2017-01-01",
        "end_time": "2020-08-01",
        "account": 100000000,
        "benchmark": BENCHMARK,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5,
        },
    },
}

# backtest and analysis
# 回测与分析
with R.start(experiment_name=EXP_NAME, recorder_id=rid, resume=True):
    # signal-based analysis
    # 基于信号的分析
    rec = R.get_recorder()
    sar = SigAnaRecord(rec)
    sar.generate()

    #  portfolio-based analysis: backtest
    # 基于投资组合的分析：回测
    par = PortAnaRecord(rec, port_analysis_config, "day")
    par.generate()

# Loading results & Analysis
# 加载结果与分析

## loading data
## 加载数据
Because Qlib leverage MLflow to save model & data.
因为 Qlib 利用 MLflow 来保存模型和数据。
All the data can be access by `mlflow ui`
所有的数据都可以通过 `mlflow ui` 进行访问。

In [ ]:
# load recorder
# 加载记录器
recorder = R.get_recorder(recorder_id=rid, experiment_name=EXP_NAME)

In [ ]:
# load previous results
# 加载先前的结果
pred_df = recorder.load_object("pred.pkl")
report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

In [ ]:
# Previous Model can be loaded. but it is not used.
# 先前的模型可以被加载，但在这里没有使用。
loaded_model = recorder.load_object("trained_model")
loaded_model

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position

## analysis position
## 持仓分析

### report
### 报告

In [ ]:
analysis_position.report_graph(report_normal_df)

### risk analysis
### 风险分析

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

## analysis model
## 模型分析

In [ ]:
label_df = dataset.prepare("test", col_set="label")
label_df.columns = ["label"]

### score IC
### 得分 IC（信息系数）

In [ ]:
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
analysis_position.score_ic_graph(pred_label)

### model performance
### 模型性能

In [ ]:
analysis_model.model_performance_graph(pred_label)